# Nilufer Bina Metrekare Birim Degerleri - LOKAL Indirme

Bu notebook **Databricks'te DEGIL, kendi bilgisayarinda (lokal Jupyter/VS Code)**
calistirilmak icin yazilmistir. `dbutils`/`spark` kullanmaz.

Neden lokal: `acikveri.nilufer.bel.tr` yalnizca Turkiye IP'lerinden erisime
acik gorunuyor; Databricks cluster'i (Azure, Turkiye disi bolge) bu siteye
baglanirken connection timeout aliyor. Bu notebook'u kendi (Turkiye IP'li)
makinende calistirip veriyi indir, sonra ciktiyi Databricks'e (DBFS/Volume)
manuel yukleyip `Nilufer Bina Datalake Upload.ipynb` ile datalake'e gonder.

`acikveri.nilufer.bel.tr` uzerindeki CKAN API'sini kullanir, 'Bina
Metrekare Birim Degerleri' veri setinin (tum Nilufer icin, mahalle bazli
degil - insaat turu/sinifi/sekli x yil bazli) en guncel kaynak dosyasini
bulur, indirir ve temizlenmis bir CSV olarak kaydeder.

Nazik/performansli yaklasim:
- Sadece 1 API cagrisi (`package_show`) yapilir, sadece metadata icin.
- Asil veri, `/api/` altinda olmayan resmi kaynak indirme URL'inden tek
  seferde indirilir (robots.txt'teki `Disallow: /api/` kapsamina girmez).
- Paralel/tekrarli istek yoktur, `Crawl-Delay: 10` kuralina saygi gosterilir.
- Lisans: CC BY 4.0 (Nilufer Belediyesi Acik Veri Lisansi) - kullanirken atif gerekir.

In [ ]:
%pip install pandas requests openpyxl

In [ ]:
import os
import time

import pandas as pd
import requests

CKAN_BASE_URL = "https://acikveri.nilufer.bel.tr"
DATASET_ID = "bina-metrekare-birim-degerleri"

REQUEST_HEADERS = {
    "User-Agent": "aXet-Project/1.0 (acik veri arastirma amacli)",
    "Accept": "application/json",
}

In [ ]:
def get_ckan_resource(base_url, dataset_id, request_headers, preferred_format="XLSX"):

    url = f"{base_url}/api/3/action/package_show"

    response = requests.get(
        url,
        params={"id": dataset_id},
        headers=request_headers,
        timeout=30,
    )
    response.raise_for_status()

    resources = response.json()["result"]["resources"]

    for resource in resources:
        if resource.get("format", "").upper() == preferred_format:
            return resource

    return resources[0] if resources else None


def download_file(url, dest_path, request_headers=None, chunk_size=1024 * 64):

    with requests.get(url, headers=request_headers, stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(dest_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                f.write(chunk)

    return dest_path

In [ ]:
resource = get_ckan_resource(CKAN_BASE_URL, DATASET_ID, REQUEST_HEADERS)

print("Bulunan kaynak:", resource["name"])
print("Format:", resource["format"])
print("Son guncelleme:", resource.get("last_modified"))

raw_path = "bina_birim_degerleri_raw.xlsx"

download_file(resource["url"], raw_path, request_headers=REQUEST_HEADERS)

time.sleep(1)

print("Indirildi:", raw_path)

In [ ]:
df_raw = pd.read_excel(raw_path)

os.remove(raw_path)

df = df_raw[["Yil", "InsaatTurAdi", "InsaatSinifAdi", "InsaatSekliAdi", "BinaBirimDeger"]].copy()

df.columns = ["yil", "insaat_tur_adi", "insaat_sinif_adi", "insaat_sekli_adi", "bina_birim_degeri"]

df["yil"] = pd.to_numeric(df["yil"], errors="coerce")
df["bina_birim_degeri"] = pd.to_numeric(df["bina_birim_degeri"], errors="coerce")

print("Tum Nilufer, tum yillar (filtre yok):", df.shape)
print("Yil araligi:", df["yil"].min(), "-", df["yil"].max())
df.head(10)

In [ ]:
output_path = "nilufer_bina_birim_degerleri.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Kaydedildi: {output_path}")
print("Sirada: Bu CSV dosyasini Databricks'e (DBFS veya Unity Catalog Volume) manuel yukle,")
print("ardindan 'Nilufer Bina Datalake Upload.ipynb' notebook'unu o path ile calistir.")